# System Design: Part 2
## Caching, Databases & CAP Theorem

**Topics covered:**
1. Why Caching Exists
2. Caching Strategies (Cache-aside, Write-through, Write-back)
3. Cache Invalidation & Eviction
4. Redis Fundamentals
5. SQL vs NoSQL Trade-offs
6. Database Replication
7. CAP Theorem

---
> **Context:** This notebook is the "data layer" half of system design. How you store, retrieve, and cache data is usually where performance is won or lost.

## 1. Why Caching Exists

The core problem: **databases are slow compared to memory**.

| Storage | Latency |
|---|---|
| CPU cache (L1) | ~1 nanosecond |
| RAM | ~100 nanoseconds |
| SSD (local) | ~100 microseconds |
| Database (same network) | ~1–10 milliseconds |
| Database (cross-region) | ~100+ milliseconds |

A cache stores **frequently accessed data in a faster layer** so you don't hit the database every time.

```
Without cache:  Request → DB every time         → slow
With cache:     Request → Cache (hit) → return  → fast
                Request → Cache (miss) → DB → Cache → return
```

### The 80/20 Rule of Caching

In most apps, **80% of traffic reads the same 20% of data**. Caching that 20% eliminates most of your database load.

Classic examples:
- A news site: The top 10 articles get 80% of traffic → cache those
- An e-commerce site: The homepage, popular products → cache those
- A social app: Logged-in user's profile, feed → cache per user

---
## 2. Caching Strategies

### Strategy 1: Cache-Aside (Lazy Loading)

The application code manages the cache manually. Most common pattern.

**Read path:**
```
1. Check cache
2. If HIT → return cached data
3. If MISS → query database → store in cache → return data
```

**Write path:**
```
1. Write to database
2. Invalidate (delete) the cache entry
   (don't update cache — let it be repopulated on next read)
```

✅ **Pros:** Only caches what's actually requested. Cache failure doesn't break the system.  
❌ **Cons:** First request after a miss is slow. Cache can get stale if you forget to invalidate.

---

### Strategy 2: Write-Through

Every write goes to **both cache AND database** at the same time.

```
Write → Cache + DB (simultaneously)
Read  → Cache (always fresh)
```

✅ **Pros:** Cache is always in sync with DB. No stale reads.  
❌ **Cons:** Every write is slower (two writes). Cache fills up with data that may never be read.

---

### Strategy 3: Write-Back (Write-Behind)

Write to cache immediately. Flush to database **asynchronously** (later).

```
Write → Cache only (return immediately)
Background: Cache → DB (every X seconds, or when evicted)
```

Think of it like taking notes on a whiteboard during a meeting — fast to write, but if the room floods before you transfer them to paper, they're gone.

Real use case: game leaderboard scores, analytics counters, shopping cart updates — where you need to handle thousands of writes per second and can tolerate a tiny window of potential loss.

✅ **Pros:** Fastest writes. Great for write-heavy workloads.  
❌ **Cons:** **Data loss risk** — if cache crashes before flushing to DB, that data is permanently gone. Complex to implement correctly.

---

### Strategy 4: Read-Through

The cache sits in front of the DB and handles misses itself — the app only ever talks to the cache, never the DB directly.

```
App → Cache
         ↓ (on miss, cache fetches from DB automatically)
        DB → Cache stores it → App
```

Vs Cache-Aside: in cache-aside, **your app code** checks cache, calls DB on miss, then populates cache manually. In read-through, **the cache layer does all of that for you** — your app just asks the cache and gets an answer regardless.

Useful when you want a clean separation — app logic doesn't need to know anything about the DB. Less common for custom apps since it requires a cache client/library that supports it (e.g. AWS DAX for DynamoDB).

✅ **Pros:** Cleaner app code — single data source from the app's perspective.  
❌ **Cons:** First request still slow on miss. Less control over exactly what gets cached.

### Quick Comparison

| Strategy | Write Speed | Read Speed | Stale Risk | Data Loss Risk |
|---|---|---|---|---|
| Cache-aside | Normal | Fast (after warmup) | Medium | None |
| Write-through | Slower | Always fast | None | None |
| Write-back | **Fastest** | Always fast | None | **High** |

In [ ]:
# Cache-Aside Pattern — implemented in pure Python (no Redis needed to understand the concept)

import time

class FakeDatabase:
    """Simulates a slow database"""
    def __init__(self):
        self._data = {1: "Alice", 2: "Bob", 3: "Carol"}
    
    def get_user(self, user_id: int) -> str:
        time.sleep(0.1)  # Simulate DB latency (100ms)
        return self._data.get(user_id)
    
    def update_user(self, user_id: int, name: str):
        time.sleep(0.05)  # Simulate write latency
        self._data[user_id] = name


class CacheAsideService:
    def __init__(self, db: FakeDatabase):
        self.db = db
        self._cache = {}  # In real life: Redis
    
    def get_user(self, user_id: int) -> str:
        # Step 1: Check cache
        if user_id in self._cache:
            print(f"  [CACHE HIT] user {user_id}")
            return self._cache[user_id]
        
        # Step 2: Cache miss — go to DB
        print(f"  [CACHE MISS] user {user_id} — fetching from DB...")
        user = self.db.get_user(user_id)
        
        # Step 3: Store in cache
        self._cache[user_id] = user
        return user
    
    def update_user(self, user_id: int, name: str):
        # Write to DB first
        self.db.update_user(user_id, name)
        # Invalidate cache — don't update, let next read repopulate
        self._cache.pop(user_id, None)
        print(f"  [CACHE INVALIDATED] user {user_id}")


db = FakeDatabase()
service = CacheAsideService(db)

print("First read (cold cache):")
start = time.time()
print(service.get_user(1))
print(f"  Took: {(time.time()-start)*1000:.0f}ms\n")

print("Second read (warm cache):")
start = time.time()
print(service.get_user(1))
print(f"  Took: {(time.time()-start)*1000:.0f}ms\n")

print("Update user (invalidates cache):")
service.update_user(1, "Alice Smith")

print("\nRead after update (cache miss again):")
start = time.time()
print(service.get_user(1))
print(f"  Took: {(time.time()-start)*1000:.0f}ms")

---
## 3. Cache Invalidation & Eviction

### Cache Invalidation

The hardest problem in caching. When does cached data become stale?

Two approaches:

**TTL (Time-To-Live)** — expire data automatically after N seconds
```python
# Redis example
redis.set("user:1", "Alice", ex=300)  # expires in 5 minutes
```
- Simple. Works well when slight staleness is acceptable.
- Bad when consistency matters (bank balance, inventory count).

**Explicit invalidation** — delete cache key on write (cache-aside pattern above)
- Stronger consistency but you must not forget to invalidate.

---

### Cache Eviction Policies

Cache has limited memory. What happens when it's full? The eviction policy decides what to remove.

| Policy | How it works | Use when |
|---|---|---|
| **LRU** (Least Recently Used) | Remove the item not accessed for the longest time | Most common general-purpose choice |
| **LFU** (Least Frequently Used) | Remove the item accessed the fewest times | Good for stable popularity patterns |
| **TTL-based** | Remove expired items first | When data has natural expiry |
| **Random** | Remove a random item | Surprisingly effective, simple |

Redis defaults to LRU. This is almost always the right choice.

### Cache Stampede (Thundering Herd)

A classic problem: A popular cache key expires. Suddenly 1000 requests all hit the DB simultaneously.

Solutions:
- **Jitter** — add random offset to TTL so keys don't all expire at once
- **Mutex/Lock** — first request acquires a lock, fetches from DB; others wait for cache to warm
- **Background refresh** — proactively refresh cache before TTL expires

---
## 4. Redis Fundamentals

Redis = **Re**mote **Di**ctionary **S**erver

It's an in-memory data store. Think of it as a very fast dictionary (hash map) that lives on a separate server and can be shared by all your app servers.

### What makes Redis special?

- **In-memory** → microsecond latency (not milliseconds like disk-based DBs)
- **Supports many data types** (not just strings)
- **Atomic operations** → safe for counters, rate limiting
- **Optional persistence** → can flush to disk periodically
- **Pub/Sub messaging** → can act as a simple message broker

### Redis Data Types

| Type | Use case | Example |
|---|---|---|
| **String** | Simple cache values, counters | `SET user:1:name "Alice"` |
| **Hash** | Object with multiple fields | `HSET user:1 name Alice age 30` |
| **List** | Activity feeds, queues | `LPUSH notifications "new message"` |
| **Set** | Unique items, tags | `SADD user:1:followers 2 3 4` |
| **Sorted Set** | Leaderboards, ranked data | `ZADD leaderboard 1000 "alice"` |

### Common Redis Use Cases

**1. Session storage**
```python
redis.set(f"session:{token}", user_id, ex=3600)  # expires in 1 hour
```

**2. Rate limiting**
```python
count = redis.incr(f"rate:{user_id}:{minute}")
redis.expire(f"rate:{user_id}:{minute}", 60)
if count > 100:
    raise RateLimitExceeded()
```

**3. Leaderboard**
```python
redis.zadd("leaderboard", {"alice": 1500})
top10 = redis.zrevrange("leaderboard", 0, 9, withscores=True)
```

**4. Distributed lock**
```python
# Acquire lock (only one server can hold it)
acquired = redis.set("lock:resource", "1", ex=10, nx=True)  # nx = only if not exists
```

In [ ]:
# Redis simulation using a plain dict — same concepts, no Redis server needed
# In real code, replace FakeRedis with: import redis; r = redis.Redis()

import time
from datetime import datetime

class FakeRedis:
    """Mimics basic Redis operations for learning"""
    def __init__(self):
        self._store = {}   # key → value
        self._expiry = {}  # key → expiry timestamp
    
    def set(self, key, value, ex=None):
        self._store[key] = value
        if ex:
            self._expiry[key] = time.time() + ex
    
    def get(self, key):
        if key in self._expiry and time.time() > self._expiry[key]:
            del self._store[key]
            del self._expiry[key]
            return None
        return self._store.get(key)
    
    def incr(self, key):
        current = int(self._store.get(key, 0))
        self._store[key] = current + 1
        return current + 1
    
    def delete(self, key):
        self._store.pop(key, None)


redis = FakeRedis()

# --- Use Case 1: Simple cache ---
print("=== Simple Cache ===")
redis.set("user:1:name", "Alice", ex=5)
print(f"Get immediately: {redis.get('user:1:name')}")
time.sleep(6)
print(f"After TTL expires: {redis.get('user:1:name')}")

# --- Use Case 2: Rate limiting ---
print("\n=== Rate Limiting ===")
user_id = 42
minute = datetime.now().strftime("%Y%m%d%H%M")
rate_key = f"rate:{user_id}:{minute}"

RATE_LIMIT = 5  # max 5 requests per minute

for i in range(7):
    count = redis.incr(rate_key)
    if count > RATE_LIMIT:
        print(f"  Request {i+1}: RATE LIMITED (count={count})")
    else:
        print(f"  Request {i+1}: OK (count={count})")

---
## 5. SQL vs NoSQL Trade-offs

This is one of the most debated topics in system design. The answer is always: **it depends on your use case**.

### SQL (Relational Databases)

Examples: PostgreSQL, MySQL, SQLite

Data is stored in **tables with rows and columns**, with enforced relationships between tables.

**Strengths:**
- **ACID transactions** — Atomicity, Consistency, Isolation, Durability. Your data is always correct.
- **JOINs** — query across related tables easily
- **Schema enforcement** — data must match the defined structure
- **Mature, battle-tested** — decades of optimization

**Use SQL when:**
- Data has clear relationships (users → orders → products)
- You need transactions (banking, inventory)
- Your schema is relatively stable
- You need complex queries

---

### NoSQL Databases

"Not Only SQL" — a broad category with several types:

| Type | Examples | Data model | Best for |
|---|---|---|---|
| **Document** | MongoDB, Firestore | JSON-like documents | Flexible schemas, nested data |
| **Key-Value** | Redis, DynamoDB | key → value | Sessions, caches, simple lookups |
| **Wide-Column** | Cassandra, HBase | Columns grouped by row key | Time-series, IoT, write-heavy |
| **Graph** | Neo4j | Nodes and edges | Social networks, recommendations |

**Common NoSQL strengths:**
- **Flexible schema** — no need to define structure upfront
- **Horizontal scalability** — designed to scale out from the start
- **High write throughput** — especially Cassandra

**Common NoSQL weaknesses:**
- Weaker consistency guarantees (often "eventual consistency")
- No JOINs — you denormalize data (store duplicates)
- Less mature tooling for complex queries

---

### The Real Decision Framework

```
Need ACID transactions?          → SQL
Need complex joins/aggregations? → SQL
Schema changes frequently?       → NoSQL (Document)
Massive write throughput needed? → NoSQL (Wide-column, e.g. Cassandra)
Simple key lookups?              → Redis or DynamoDB
Social graph / recommendations?  → Graph DB
```

> **Reality check:** Most production systems use **both**. PostgreSQL for transactional data, Redis for caching, maybe Elasticsearch for search. The question isn't SQL *or* NoSQL — it's which tool for which job.

---
## 6. Database Replication

Replication means keeping **copies of your data on multiple database servers**.

### Why replicate?

1. **High availability** — if the primary DB fails, a replica takes over
2. **Read scaling** — spread read traffic across multiple replicas
3. **Geographic distribution** — keep replicas close to users in different regions

### Primary-Replica (Leader-Follower) Replication

Most common pattern:

```
              Writes
App ──────────────────→ [Primary DB]
                              │
                    replicates│ (async)
                              ▼
                    [Replica 1]  [Replica 2]
                         ▲           ▲
                         └─────┬─────┘
                               │ Reads
                              App
```

- **All writes** go to the primary
- **Reads** can go to replicas (primary or replica)
- Replicas get changes from primary asynchronously

### Replication Lag

Async replication means replicas are **slightly behind** the primary (usually milliseconds, sometimes seconds).

This causes **read-your-own-writes inconsistency**:
```
1. User updates their profile (write to primary)
2. User immediately views profile (read from replica)
3. Replica hasn't synced yet → user sees old data!
```

Solutions:
- Route a user's reads to primary immediately after they write
- Use **synchronous replication** (wait for replica to confirm before returning) — slower but no lag

### Failover

When the primary fails:
1. A replica is **promoted** to become the new primary
2. App is reconfigured to point to new primary
3. Old primary comes back online as a replica

This happens automatically in managed databases (AWS RDS, Google Cloud SQL).

### Multi-Primary (Multi-Master) Replication

Multiple nodes can accept writes. More complex, harder to handle conflicts, but better for geo-distributed writes.  
Used by: CockroachDB, Google Spanner, Cassandra.

---
## 7. CAP Theorem

The CAP theorem is one of the most important — and most misunderstood — concepts in distributed systems.

### The Theorem

In a distributed system (data on multiple machines), you can only **fully guarantee 2 of these 3 properties** at a time:

| Property | What it means |
|---|---|
| **C**onsistency | Every read gets the most recent write (or an error) |
| **A**vailability | Every request gets a response (not necessarily the latest data) |
| **P**artition Tolerance | The system keeps working even if some nodes can't communicate |

### Why you can't have all three

A **network partition** happens when two parts of your system can't communicate (a network cable fails, a datacenter goes down, etc.).

Partition tolerance is **non-negotiable** for any real distributed system — network failures happen. You can't choose to not have partition tolerance.

So the real choice is: **when a partition occurs, do you sacrifice Consistency or Availability?**

```
   Network partition occurs!
   [Node A] ✗✗✗ [Node B]
   
   A request comes in...
   
   CP choice: Return an error ("I can't guarantee fresh data")
              → Consistent, but not Available
   
   AP choice: Return possibly stale data ("Here's what I have")
              → Available, but not Consistent
```

### CP vs AP — Real Examples

**CP Systems** (choose consistency over availability during partition):
- **PostgreSQL, MySQL** — won't return stale data; may be unavailable during failover
- **HBase, Zookeeper** — return error rather than inconsistent data
- **Banking systems** — you'd rather get an error than see the wrong balance

**AP Systems** (choose availability over consistency during partition):
- **Cassandra, DynamoDB** — always responds, might be slightly stale
- **DNS** — always returns an answer, might be cached and old
- **Shopping cart** — it's fine if two devices briefly show different carts

### Eventual Consistency

AP systems use **eventual consistency**: all nodes will eventually agree on the same data — just not immediately.

```
t=0: User updates name to "Alice Smith"
t=1: Node A has "Alice Smith", Node B still has "Alice"
t=2: Replication syncs
t=3: Both nodes have "Alice Smith" ← eventually consistent
```

The window of inconsistency is usually milliseconds. For most apps, this is fine.

### Modern Nuance: PACELC

CAP only describes behavior *during* a partition. The PACELC model extends it:
- Even when there's **no** partition, there's a trade-off between **Latency** and **Consistency**.
- Synchronous replication → strong consistency, higher latency
- Async replication → eventual consistency, lower latency

This is the trade-off that matters day-to-day (partitions are rare; this latency vs consistency choice is constant).

In [ ]:
# Simulating eventual consistency to build intuition

import time
import threading

class Node:
    def __init__(self, name: str):
        self.name = name
        self.data = {}
        self.peers = []
    
    def write(self, key: str, value: str, replicate: bool = True):
        self.data[key] = value
        print(f"  [{self.name}] Wrote {key}={value}")
        
        if replicate:
            def replicate_async():
                time.sleep(0.2)  # Simulate replication lag (200ms)
                for peer in self.peers:
                    peer.write(key, value, replicate=False)
                    print(f"  [{peer.name}] Replicated {key}={value} (from {self.name})")
            
            threading.Thread(target=replicate_async).start()
    
    def read(self, key: str) -> str:
        return self.data.get(key, "<not yet replicated>")


# Set up two nodes
node_a = Node("Node A (primary)")
node_b = Node("Node B (replica)")
node_a.peers = [node_b]

print("=== Eventual Consistency Demo ===")
print("\nWriting to Node A (primary)...")
node_a.write("username", "Alice Smith")

print("\nReading immediately from both nodes:")
print(f"  Node A: {node_a.read('username')}")
print(f"  Node B: {node_b.read('username')}  ← stale! replication hasn't happened yet")

print("\nWaiting for replication...")
time.sleep(0.5)

print("\nReading after replication lag:")
print(f"  Node A: {node_a.read('username')}")
print(f"  Node B: {node_b.read('username')}  ← now consistent!")

---
## Summary

### Caching

| Strategy | When to use |
|---|---|
| **Cache-aside** | Default choice. App controls cache, only stores what's needed |
| **Write-through** | When you need cache always in sync with DB |
| **Write-back** | High write throughput (analytics, counters). Accept data loss risk |

**Cache eviction:** LRU is the default. Use TTL for data with natural expiry.

---

### Redis

- In-memory, microsecond latency
- Use for: sessions, rate limiting, leaderboards, distributed locks, caching
- Supports: Strings, Hashes, Lists, Sets, Sorted Sets

---

### SQL vs NoSQL

| Need | Use |
|---|---|
| ACID transactions, complex queries | SQL (PostgreSQL) |
| Flexible schema, JSON documents | Document DB (MongoDB) |
| High write throughput | Wide-column (Cassandra) |
| Fast key-value lookups | Key-value (Redis, DynamoDB) |
| Graph relationships | Graph DB (Neo4j) |

---

### CAP Theorem

```
Partition tolerance is required in any distributed system.
Real choice: CP or AP when a partition occurs.

CP → return error, never stale data   (banks, inventory)
AP → return possibly stale data       (social feeds, DNS, carts)
```

**Eventual consistency** = AP systems will sync up, just not immediately.

---

### The Full Picture

A typical production system at scale:

```
[Client]
   ↓
[CDN] ← static assets cached globally
   ↓
[Load Balancer]
   ↓
[Stateless API Servers] ← horizontal scaling
   ↓           ↓
[Redis Cache]  [Message Queue]
   ↓
[Primary DB] → [Read Replicas] ← reads spread across replicas
```

Every arrow is a design decision with trade-offs. Now you understand what those trade-offs are.